build a ANN network to as contrastive objector for DANN

In [1]:
import torch.utils.data as data
import os
import scipy.io as sio
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np
import random
import torch.nn as nn
from reg_functions import reg_indicator
import torch.optim as optim
import torch.backends.cudnn as cudnn

In [2]:
class GetLoader(data.Dataset):
    def __init__(self, data_root, data_label_root, transform=None):
        self.data_root = os.path.join("./Data", data_root)
        self.data_label_root = os.path.join("./Data", data_label_root)
        self.transform = transform
        
        # load data
        self.data = sio.loadmat(self.data_root)['data']
        self.data_label = sio.loadmat(self.data_label_root)['EQvec']
        
        # transform the data and labels: standardize
        if self.transform:
            scaler = StandardScaler()
            self.data = scaler.fit_transform(self.data)
        
    def __getitem__(self, item):
        d = self.data[item]
        l = self.data_label[item]
        
        # convert to tensor
        d = torch.tensor(d, dtype=torch.float32)
        l = torch.tensor(l, dtype=torch.float32)
        
        return d, l
    
    def __len__(self):
        return len(self.data)

In [3]:
class ANN(nn.Module):

    def __init__(self):
        super(ANN, self).__init__()
        self.feature = nn.Sequential()
        self.feature.add_module('f_linear1', nn.Linear(15, 64))
        self.feature.add_module('f_relu1', nn.ReLU(True))
        self.feature.add_module('f_linear2', nn.Linear(64, 128))
        self.feature.add_module('f_relu2', nn.ReLU(True))

        self.value_regressor = nn.Sequential()
        self.value_regressor.add_module('v_linear1', nn.Linear(128, 64))
        self.value_regressor.add_module('v_relu1', nn.ReLU(True))
        self.value_regressor.add_module('v_linear2', nn.Linear(64, 1))

    def forward(self, input_data):
        feature = self.feature(input_data)
        reg_output = self.value_regressor(feature)

        return reg_output

In [4]:
def train(model, train_loader, optimizer, criterion, epoch, device):
    model.train()
    loss_result = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss_result += loss.item()
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    loss_result /= len(train_loader.dataset)
    return loss_result
        

In [5]:
def test(model, test_loader, criterion, epoch, device):
    model.eval()
    test_loss_result = 0
    r2_result = 0
    rmse_result = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss_result += criterion(output, target).item()  # sum up batch loss
            r2, rmse = reg_indicator(output, target)
            r2_result += r2
            rmse_result += rmse

    test_loss_result /= len(test_loader.dataset)
    r2_result /= len(test_loader.dataset)
    rmse_result /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}\n'.format(test_loss_result))
    print('Test set: Average R2: {:.4f}\n'.format(r2_result))
    print('Test set: Average RMSE: {:.4f}\n'.format(rmse_result))
    return test_loss_result, r2_result, rmse_result

In [6]:
train_dataset_name = 'X_kla120.mat'
train_dataset_labels_name = 'EQvec_kla120.mat'
test_dataset_name = 'X_kla240.mat'
test_dataset_labels_name = 'EQvec_kla240.mat'

cudnn.benchmark = True
lr = 1e-3
batch_size = 128
n_epoch = 100

manual_seed = 42
random.seed(manual_seed)
torch.manual_seed(manual_seed)

# load data
train_dataset = GetLoader(train_dataset_name, train_dataset_labels_name, transform=True)
test_dataset = GetLoader(test_dataset_name, test_dataset_labels_name, transform=True)

# create dataloaders
dataloader_source = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=32)

dataloader_target = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=32)

print('read the data from the dataset')

read the data from the dataset


d:\anaconda\envs\pytorch\lib\site-packages\torch\utils\data\dataloader.py:561: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 20 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [7]:
# create model
model = ANN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    model = model.cuda()

# create optimizer
optimizer = optim.Adam(model.parameters(), lr=lr)

# create loss function
criterion = nn.MSELoss()

In [8]:
# train model and test model
train_loss_list = []
test_loss_list = []
test_r2_list = []
test_rmse_list = []
for epoch in range(1, n_epoch + 1):
    train_loss = train(model, dataloader_source, optimizer, criterion, epoch, device)
    test_loss, r2, rmse = test(model, dataloader_target, criterion, epoch, device)
    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)
    test_r2_list.append(r2)
    test_rmse_list.append(rmse)

In [ ]:
import matplotlib.pyplot as plt
# draw the loss curve
plt.plot(train_loss_list, label='train_loss')
plt.plot(test_loss_list, label='test_loss')
plt.legend()
plt.show()
plt.savefig('figures/ANN_loss.png')

# draw the r2 curve
plt.plot(test_r2_list, label='test_r2')
plt.legend()
plt.show()
plt.savefig('figures/ANN_r2.png')

# draw the rmse curve
plt.plot(test_rmse_list, label='test_rmse')
plt.legend()
plt.show()
plt.savefig('figures/ANN_rmse.png')